In [118]:
import requests
import polars as pl


In [119]:
data = requests.get("https://ogd-static.voteinfo-app.ch/v4/ogd/proporz_resultate_2026_03_08.json").json()

In [120]:
results = list(filter(lambda row: row['vorlagenId']==377642, data['kantone'][1]['vorlagen']))[0]


In [121]:
list(results['zaehlkreise'][0]['resultat']['listen'])


[{'listeNummer': '01',
  'listeCode': 'SP',
  'stimmen': None,
  'waehler': None,
  'waehlerProzent': None,
  'letzteWahlWaehlerProzent': None,
  'gewinnWaehlerProzent': None},
 {'listeNummer': '02',
  'listeCode': 'FDP',
  'stimmen': None,
  'waehler': None,
  'waehlerProzent': None,
  'letzteWahlWaehlerProzent': None,
  'gewinnWaehlerProzent': None},
 {'listeNummer': '03',
  'listeCode': 'Grüne',
  'stimmen': None,
  'waehler': None,
  'waehlerProzent': None,
  'letzteWahlWaehlerProzent': None,
  'gewinnWaehlerProzent': None},
 {'listeNummer': '04',
  'listeCode': 'GLP',
  'stimmen': None,
  'waehler': None,
  'waehlerProzent': None,
  'letzteWahlWaehlerProzent': None,
  'gewinnWaehlerProzent': None},
 {'listeNummer': '05',
  'listeCode': 'SVP',
  'stimmen': None,
  'waehler': None,
  'waehlerProzent': None,
  'letzteWahlWaehlerProzent': None,
  'gewinnWaehlerProzent': None},
 {'listeNummer': '06',
  'listeCode': 'AL',
  'stimmen': None,
  'waehler': None,
  'waehlerProzent': None,
 

In [122]:
mapping_str = """SP
FDP
Grüne
GLP
SVP
AL
Die Mitte
EVP
Andere"""

mapping = list(mapping_str.split('\n'))

In [123]:

rows = []
for zaehlkreis in results['zaehlkreise']:
    for liste in zaehlkreis['resultat']['listen']:
        rows.append({
            "zaehlkreis": zaehlkreis['geoLevelname'],
            "liste": liste['listeCode'] if liste['listeCode'] in mapping else "Andere",
            "stimmen": liste['stimmen']
        })

df = pl.DataFrame(rows)

In [124]:
df_wide = df.pivot(index="liste", on="zaehlkreis", values="stimmen", aggregate_function="sum")

In [125]:
df_order = pl.DataFrame({'liste': mapping})

df_final = df_order.join(df_wide, on='liste', how='left')

In [126]:
df_final

liste,Zürich Kreise 1 und 2,Zürich Kreis 3,Zürich Kreis 4 und 5,Zürich Kreis 6,Zürich Kreise 7 und 8,Zürich Kreis 9,Zürich Kreis 10,Zürich Kreis 11,Zürich Kreis 12
str,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""SP""",0,74242,0,0,0,0,57801,104439,0
"""FDP""",0,34592,0,0,0,0,29571,52262,0
"""Grüne""",0,23312,0,0,0,0,19193,34397,0
"""GLP""",0,22310,0,0,0,0,21351,41061,0
"""SVP""",0,17543,0,0,0,0,19409,64198,0
"""AL""",0,18943,0,0,0,0,11432,13309,0
"""Die Mitte""",0,9043,0,0,0,0,7169,20337,0
"""EVP""",0,1858,0,0,0,0,2388,8882,0
"""Andere""",0,4016,0,0,0,0,1999,4310,0


In [127]:
df_final.write_excel("results_gr.xlsx")